# Módulo 04 · Aula 1 — Bancos relacionais e o `SELECT`

**Capacitação Introdutória de Ciência de Dados · FEA.dev**

---

Até aqui, todo dado desta capacitação chegou até você como um arquivo: um `.csv`, um
`.xlsx`. Você abriu com `pd.read_csv`, e a análise começou.

Fora daqui, quase nunca é assim. Os dados de uma empresa não moram em arquivos soltos —
moram em um **banco de dados**, e a linguagem para conversar com ele é o **SQL**. É a
habilidade que aparece em praticamente toda descrição de vaga de dados, e é o passo que
vem *antes* do `read_csv` no mundo real: alguém escreveu uma consulta para produzir aquele
arquivo.

Esta aula é a porta de entrada. Ao final dela você vai:

- entender o que é um banco relacional e por que os dados são guardados assim;
- conectar-se a um banco pelo Python e trazer o resultado para um DataFrame;
- escrever consultas com `SELECT`, `WHERE`, `ORDER BY` e `LIMIT`;
- lidar com `NULL` sem cair na armadilha clássica;
- entender em que ordem o banco realmente executa a sua consulta.

**Tempo estimado:** 75 minutos.

> **Boa notícia:** você já sabe quase tudo isto. No módulo 02 você filtrou linhas com
> máscara booleana, ordenou com `sort_values` e pegou as primeiras com `head`. São as
> mesmas três operações. Muda a língua, não a ideia.

## 1. Por que SQL

Um arquivo CSV funciona bem até certo ponto. Depois dele, três coisas quebram.

**Tamanho.** `acoes_b3.csv` tem 9.968 linhas e cabe na memória sem esforço. A tabela de
transações de uma corretora tem bilhões. Você não vai carregar isso num DataFrame — vai
pedir ao banco só o pedaço que interessa, e o banco foi construído para achar esse pedaço
rápido.

**Concorrência.** Um arquivo tem um dono. Um banco atende centenas de pessoas ao mesmo
tempo, sem que a análise de uma corrompa a leitura da outra.

**Integridade.** Em um CSV, nada impede que apareça uma cotação de um ticker que não
existe no cadastro. Em um banco bem modelado, isso é *proibido* — o banco recusa. Você viu
no módulo 02 o trabalho que dá limpar dados sujos; boa parte desse trabalho é evitável se
a sujeira nunca entra.

**SQL** (*Structured Query Language*) é a linguagem para pedir dados a esse banco. Ela tem
cinquenta anos, é padronizada, e é a mesma no Postgres, no SQL Server, no Oracle, no
BigQuery e no SQLite que vamos usar aqui. Aprender uma vez serve para todos — as diferenças
existem, mas são detalhes de sotaque, e a aula 5 trata delas.

## 2. O modelo relacional

Um banco relacional guarda os dados em **tabelas**. Uma tabela é exatamente o que você já
conhece de um DataFrame: linhas e colunas, com nome e tipo.

O que é novo — e é a ideia central — é que os dados ficam **repartidos em várias tabelas
que se referenciam**.

Pense no `acoes_b3.csv`. Se ele guardasse também o nome e o setor de cada empresa, o texto
`"Petróleo Brasileiro S.A. - Petrobras"` apareceria repetido em 1.246 linhas. Isso
desperdiça espaço, mas o problema real é outro: se a razão social mudar, você tem 1.246
lugares para corrigir, e basta esquecer um para a base ficar inconsistente.

A solução é separar:

| Tabela | Uma linha é... | Guarda |
|---|---|---|
| `empresas` | uma empresa | o que **não muda** por pregão: nome, setor, controle |
| `cotacoes` | um pregão de um papel | o que muda todo dia: preços, volume |

E as duas se conectam por uma coluna em comum, o `ticker`. Isso se chama **normalização**,
e é a razão de os dados chegarem sempre quebrados em pedaços. Juntar os pedaços de volta é
o `JOIN`, assunto da aula 3.

### Chaves

| Conceito | O que é | No nosso banco |
|---|---|---|
| **Chave primária** | a coluna (ou conjunto) que identifica uma linha de forma única | `empresas.ticker`; em `cotacoes`, o par `(data, ticker)` |
| **Chave estrangeira** | uma coluna que aponta para a chave primária de outra tabela | `cotacoes.ticker` → `empresas.ticker` |

A chave estrangeira é o que garante a integridade: o banco recusa uma cotação de um ticker
que não exista em `empresas`.

## 3. O banco desta capacitação

O arquivo `data/capacitacao.db` é um banco **SQLite** com quatro tabelas, construídas a
partir dos mesmos CSVs que você já usou nos módulos 02 e 03.

Isso é proposital: **a mesma pergunta, respondida em pandas e em SQL, tem que dar o mesmo
número.** É assim que você confere se entendeu.

| Tabela | Linhas | Uma linha é |
|---|---|---|
| `empresas` | 8 | uma empresa listada |
| `cotacoes` | 9.968 | um pregão de um papel |
| `indicadores` | 60 | um mês de IPCA, Selic e dólar |
| `clientes` | 400 | um cliente da corretora fictícia — **suja de propósito** |

> **SQLite** é um banco que cabe em um arquivo. Não tem servidor, não tem senha, não tem
> instalação: o `sqlite3` já vem com o Python. Isso o torna ideal para aprender — e ele é,
> de longe, o banco mais implantado do mundo, porque está dentro de todo celular e todo
> navegador. O SQL que você vai escrever aqui é o mesmo de um Postgres de produção.

### Antes de começar — se você está no Google Colab

Este notebook lê o banco de dados da pasta `data/` do repositório, e no Colab a máquina começa vazia. **Execute a célula abaixo antes de qualquer outra**: ela traz o repositório e entra na pasta deste módulo, de modo que os caminhos `../data/...` usados no material funcionem sem alteração.

No VS Code ou no Jupyter local a célula não faz nada — os arquivos já estão no seu disco.

In [ ]:
# Setup do Google Colab.
# Traz o repositório da capacitação e entra na pasta deste módulo, para que os
# caminhos "../data/..." usados no material funcionem sem nenhuma alteração.
# Fora do Colab (VS Code, Jupyter local) esta célula não faz nada.
# Pode ser executada mais de uma vez sem problema.
import os
import subprocess
import sys

PASTA_DESTE_MODULO = "04_SQL"
REPOSITORIO = "https://github.com/gustavokatsuo/Introducao-a-Ciencia-de-Dados.git"

if "google.colab" in sys.modules and not os.path.isdir("../data"):
    destino = "/content/Introducao-a-Ciencia-de-Dados"
    if not os.path.isdir(destino):
        print("Baixando o material da capacitação...")
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORIO, destino], check=True)
    os.chdir(os.path.join(destino, PASTA_DESTE_MODULO))
    print("Pronto. Pasta de trabalho:", os.getcwd())

## 4. Conectando pelo Python

São duas peças:

1. **`sqlite3`**, da biblioteca padrão do Python, abre o arquivo e mantém a *conexão*;
2. **`pandas.read_sql_query`** manda uma consulta por essa conexão e devolve o resultado
   **já como DataFrame**.

A segunda peça é a que importa para nós: o resultado de uma consulta SQL vira exatamente o
objeto que você passou dois módulos aprendendo a manipular.

In [ ]:
import sqlite3

import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

conexao = sqlite3.connect("../data/capacitacao.db")

# Um atalho para o resto da aula: manda a consulta e devolve um DataFrame.
def consultar(sql):
    return pd.read_sql_query(sql, conexao)


print("Conectado.")

### O que existe dentro do banco?

Todo banco sabe se descrever. No SQLite, a lista de tabelas está em uma tabela interna
chamada `sqlite_master` — e ela se consulta com SQL, como qualquer outra.

In [ ]:
consultar("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")

In [ ]:
# E as colunas de uma tabela, com tipo e obrigatoriedade:
consultar("PRAGMA table_info(cotacoes)")

A coluna `notnull` mostra quais colunas não aceitam vazio, e `pk` marca a chave
primária — repare que `data` e `ticker` têm `pk` igual a 1 e 2: juntas, elas formam a
chave.

> `PRAGMA` é um comando específico do SQLite. Em outros bancos a mesma informação vem de
> `INFORMATION_SCHEMA.COLUMNS`. É o tipo de diferença de sotaque que a aula 5 discute.

## 5. `SELECT`: escolher colunas

A consulta mais simples possível tem duas partes: **o que** você quer (`SELECT`) e **de
onde** (`FROM`).

In [ ]:
consultar("SELECT ticker, empresa, setor FROM empresas")

O `*` traz todas as colunas. É ótimo para explorar e ruim em qualquer outra
situação: traz dados que você não vai usar, e a consulta quebra silenciosamente de
significado se alguém acrescentar uma coluna na tabela.

In [ ]:
consultar("SELECT * FROM empresas")

### Apelidos com `AS`

`AS` renomeia uma coluna no resultado. Serve para dar nome legível a expressões — e é
obrigatório na prática, porque ninguém quer uma coluna chamada
`ROUND(fechamento_ajustado * volume, 2)`.

In [ ]:
consultar("""
    SELECT
        ticker,
        empresa       AS razao_social,
        ano_fundacao  AS fundacao
    FROM empresas
""")

> **Sobre o estilo.** Consultas de uma linha viram consultas de vinte linhas
> rápido. A convenção que vamos usar — palavras-chave em maiúsculas, uma cláusula por
> linha, colunas indentadas — não é firula: é o que torna possível achar o erro em uma
> consulta grande. Adote desde a primeira.

## 6. Colunas calculadas

Assim como você criou colunas novas em um DataFrame, o `SELECT` aceita expressões.

In [ ]:
consultar("""
    SELECT
        ticker,
        data,
        fechamento,
        maxima - minima                            AS amplitude,
        ROUND((maxima - minima) / fechamento * 100, 2) AS amplitude_pct
    FROM cotacoes
    LIMIT 5
""")

Repare no `LIMIT 5` no final: sem ele, isso traria as 9.968 linhas. Ao explorar
uma tabela que você não conhece, **sempre** comece com `LIMIT`. É o `head()` do SQL, e em
uma tabela de produção com bilhões de linhas é a diferença entre uma resposta imediata e
travar o banco da empresa.

## 7. `WHERE`: filtrar linhas

É a máscara booleana do módulo 02, com outra sintaxe.

In [ ]:
consultar("""
    SELECT ticker, empresa, setor, ano_fundacao
    FROM empresas
    WHERE setor = 'Financeiro'
""")

> **Aspas simples para texto.** Em SQL, `'Financeiro'` é um valor de texto e
> `"Financeiro"` é (na maioria dos bancos) o *nome de uma coluna*. O SQLite é tolerante e
> aceita os dois, o que é uma armadilha: o hábito aprendido aqui quebra no Postgres. Use
> sempre aspas simples para texto.

### Os operadores

| Operador | O que faz | Em pandas |
|---|---|---|
| `=` | igual — **um sinal só** | `==` |
| `<>` ou `!=` | diferente | `!=` |
| `<`, `<=`, `>`, `>=` | comparação | igual |
| `AND`, `OR`, `NOT` | combinação | `&`, `\|`, `~` |
| `BETWEEN a AND b` | intervalo, **inclusive nas pontas** | `.between()` |
| `IN (...)` | está na lista | `.isin()` |
| `LIKE` | padrão de texto | `.str.contains()` |
| `IS NULL` | é vazio | `.isna()` |

Duas diferenças que pegam quem vem do Python:

- **`=` compara.** Em SQL não existe `==`; o `=` sozinho já é comparação, porque SQL não
  tem atribuição de variável dentro de uma consulta.
- **`AND`/`OR` por extenso**, e sem a exigência de parênteses que o pandas impõe.

In [ ]:
consultar("""
    SELECT ticker, data, fechamento, volume
    FROM cotacoes
    WHERE ticker = 'PETR4'
      AND data BETWEEN '2025-01-01' AND '2025-01-10'
    ORDER BY data
""")

Repare que comparamos datas como se fossem texto — e funcionou. Isso não é
sorte: o SQLite **não tem tipo de data**, guarda tudo como texto, e o formato ISO
(`AAAA-MM-DD`) foi escolhido justamente porque a ordem alfabética coincide com a ordem
cronológica. `'2025-01-10' > '2025-01-09'` é verdade tanto para o calendário quanto para o
dicionário.

É por isso que o script que gerou este banco converteu todas as datas para ISO. Se o
formato fosse `10/01/2025`, essa comparação daria errado silenciosamente.

In [ ]:
# IN e uma lista de valores
consultar("""
    SELECT ticker, empresa, setor
    FROM empresas
    WHERE ticker IN ('PETR4', 'VALE3', 'ITUB4')
""")

In [ ]:
# LIKE: % é "qualquer sequência", _ é "um caractere qualquer"
consultar("""
    SELECT ticker, empresa
    FROM empresas
    WHERE empresa LIKE '%S.A.%'
""")

## 8. `NULL`: a armadilha

`NULL` é o vazio do SQL — o `NaN` do pandas. E ele tem uma regra que derruba todo mundo
uma vez:

> **`NULL` não é igual a nada. Nem a si mesmo.**

`NULL = NULL` não dá verdadeiro; dá `NULL`. A lógica é que `NULL` significa "não sei", e
duas coisas que você não sabe não são necessariamente iguais.

Consequência prática: `WHERE idade = NULL` **nunca** retorna linha nenhuma. Não dá erro —
apenas devolve vazio, o que é muito pior, porque parece uma resposta. O certo é
`IS NULL`.

In [ ]:
# O jeito ERRADO: não dá erro, devolve zero linhas
consultar("SELECT COUNT(*) AS linhas FROM clientes WHERE idade = NULL")

In [ ]:
# O jeito certo
consultar("SELECT COUNT(*) AS linhas FROM clientes WHERE idade IS NULL")

21 linhas — exatamente os mesmos 21 vazios que você contou com `.isna().sum()` no
módulo 02. É o mesmo dado.

A mesma armadilha aparece na negação: `WHERE perfil_investidor <> 'Moderado'` **exclui**
as linhas onde o perfil é `NULL`, porque `NULL <> 'Moderado'` também é `NULL`, e não
verdadeiro. Se você quer os vazios junto, precisa pedir explicitamente.

In [ ]:
consultar("""
    SELECT
        COUNT(*)                                                       AS total,
        SUM(CASE WHEN perfil_investidor =  'Moderado' THEN 1 ELSE 0 END) AS igual_moderado,
        SUM(CASE WHEN perfil_investidor <> 'Moderado' THEN 1 ELSE 0 END) AS diferente_moderado,
        SUM(CASE WHEN perfil_investidor IS NULL       THEN 1 ELSE 0 END) AS sem_perfil
    FROM clientes
""")

Olhe a aritmética: 130 + 248 = 378, e não 400. Faltam 22 — exatamente os vazios.

Ou seja: `= 'Moderado'` e `<> 'Moderado'` **não cobrem a tabela inteira**. Na lógica do dia
a dia, "igual" e "diferente" esgotam as possibilidades; na lógica de três valores do SQL,
existe um terceiro grupo que não cai em nenhum dos dois. Se você contar só os "diferentes"
achando que pegou todo mundo que não é moderado, perdeu 22 clientes sem receber aviso
nenhum.

O `CASE WHEN` que apareceu aqui é assunto da aula 2; por ora, basta ver o efeito.

## 9. `ORDER BY` e `LIMIT`

`ORDER BY` ordena; `DESC` inverte. `LIMIT` corta.

In [ ]:
consultar("""
    SELECT ticker, data, volume
    FROM cotacoes
    ORDER BY volume DESC
    LIMIT 10
""")

Dá para ordenar por mais de uma coluna, com critérios de desempate — e o
critério pode até ser uma coluna que não está no `SELECT`.

In [ ]:
consultar("""
    SELECT ticker, empresa, setor, ano_fundacao
    FROM empresas
    ORDER BY setor ASC, ano_fundacao DESC
""")

## 10. `DISTINCT`: valores únicos

O `.unique()` do pandas.

In [ ]:
consultar("SELECT DISTINCT setor FROM empresas ORDER BY setor")

In [ ]:
# Em mais de uma coluna, DISTINCT vale para a COMBINAÇÃO
consultar("SELECT DISTINCT setor, controle FROM empresas ORDER BY setor")

## 11. Em que ordem o banco executa

Você escreve nesta ordem:

```sql
SELECT   colunas
FROM     tabela
WHERE    filtro
ORDER BY ordenação
LIMIT    n
```

Mas o banco executa em **outra**:

```
FROM  →  WHERE  →  SELECT  →  ORDER BY  →  LIMIT
```

Primeiro ele decide de onde vêm as linhas, depois joga fora as que não passam no filtro,
só então calcula as colunas pedidas, ordena o que sobrou e corta.

Isso não é curiosidade — explica um erro concreto, e uma armadilha de portabilidade.

**Pela ordem de execução, um apelido criado no `SELECT` não existe ainda quando o `WHERE`
roda.** Logo, filtrar por ele deveria ser um erro. Vamos ver o que acontece.

In [ ]:
consultar("""
    SELECT ticker, data, maxima - minima AS amplitude
    FROM cotacoes
    WHERE amplitude > 5
    ORDER BY amplitude DESC
    LIMIT 3
""")

**Funcionou — e é exatamente por isso que este trecho está aqui.**

O SQLite é permissivo e resolve o apelido para você. O padrão SQL não permite, e
**PostgreSQL, SQL Server e Oracle recusam essa consulta com erro.** Se você aprender o
hábito aqui, ele quebra no primeiro banco de produção em que encostar — e o erro vai vir
sem nenhuma pista de que a culpa foi um atalho que o SQLite deixava passar.

A forma portátil repete a expressão no `WHERE`. É mais feia, e é a que funciona em
qualquer banco:

In [ ]:
consultar("""
    SELECT ticker, data, maxima - minima AS amplitude
    FROM cotacoes
    WHERE maxima - minima > 5
    ORDER BY amplitude DESC
    LIMIT 3
""")

Repetir expressão longa duas vezes é ruim, e existe solução melhor: a **CTE** da
aula 4, que dá nome ao cálculo uma vez só e filtra por ele depois — de forma portátil.

In [ ]:
consultar("""
    SELECT ticker, data, maxima - minima AS amplitude
    FROM cotacoes
    WHERE maxima - minima > 5
    ORDER BY amplitude DESC
    LIMIT 5
""")

**No `ORDER BY`, por outro lado, o apelido funciona em todo banco** — inclusive
nos que recusam no `WHERE`. As duas consultas acima usam `ORDER BY amplitude` e nenhuma
teria problema no Postgres.

E o motivo é só a ordem de execução: quando o `ORDER BY` roda, o `SELECT` já aconteceu e
`amplitude` existe de verdade. Guarde a regra pela causa, não pela decoreba:

| Cláusula | Roda antes ou depois do `SELECT`? | Enxerga o apelido? |
|---|---|---|
| `WHERE` | antes | não (o SQLite abre exceção; não confie) |
| `ORDER BY` | depois | sim, em qualquer banco |

## 12. O mesmo resultado, nas duas linguagens

O fechamento da aula: uma pergunta, duas respostas, e a conferência de que batem.

> *Quais foram os dez pregões de maior volume da PETR4 em 2025?*

In [ ]:
# Em SQL
via_sql = consultar("""
    SELECT data, fechamento, volume
    FROM cotacoes
    WHERE ticker = 'PETR4'
      AND data >= '2025-01-01'
    ORDER BY volume DESC
    LIMIT 10
""")
via_sql

In [ ]:
# Em pandas, a partir do CSV do módulo 02
acoes = pd.read_csv("../data/acoes_b3.csv", parse_dates=["data"])

via_pandas = (
    acoes[(acoes["ticker"] == "PETR4") & (acoes["data"] >= "2025-01-01")]
    .sort_values("volume", ascending=False)
    .head(10)
    [["data", "fechamento", "volume"]]
    .reset_index(drop=True)
)
via_pandas

In [ ]:
# A conferência que importa
import numpy as np

iguais = np.array_equal(via_sql["volume"].values, via_pandas["volume"].values)
print("As duas rotas deram o mesmo resultado:", iguais)

| Pergunta | pandas | SQL |
|---|---|---|
| quais colunas | `[["data", "volume"]]` | `SELECT data, volume` |
| de onde | `acoes` | `FROM cotacoes` |
| filtrar linhas | máscara booleana | `WHERE` |
| ordenar | `.sort_values()` | `ORDER BY` |
| primeiras n | `.head(n)` | `LIMIT n` |

Guarde esta tabela. Ela vai crescer nas próximas aulas, e é o mapa inteiro do módulo.

## 13. Recapitulando

- Dados de verdade moram em **bancos relacionais**, repartidos em tabelas que se
  referenciam por **chaves**. SQL é a linguagem para pedi-los.
- `sqlite3` abre a conexão; `pd.read_sql_query` traz o resultado **já como DataFrame**.
- A consulta básica é `SELECT` colunas `FROM` tabela `WHERE` filtro `ORDER BY` ordem
  `LIMIT` n.
- Texto vai entre **aspas simples**. `=` compara (não existe `==`).
- Datas em formato ISO (`AAAA-MM-DD`) podem ser comparadas como texto — foi por isso que o
  banco foi construído assim.
- **`NULL` não é igual a nada.** Use `IS NULL`, nunca `= NULL`. E lembre que `<>` exclui os
  vazios silenciosamente.
- O banco executa `FROM → WHERE → SELECT → ORDER BY → LIMIT`. Daí um apelido do `SELECT`
  valer no `ORDER BY` e não valer no `WHERE`.
- Ao explorar tabela grande, **sempre** comece com `LIMIT`.

**Próxima aula:** contar, somar e agrupar — `COUNT`, `SUM`, `AVG`, `GROUP BY` e `HAVING`.
É o `groupby` do módulo 02, na língua do banco.

In [ ]:
conexao.close()
print("Conexão fechada.")